# YOLO Annotatoions Generator
This notebook is used to generate YOLO annotations by using a pre-trained YOLO model and storing them in a .json file using the COCO dataset format.

End format should be:
{
  "images": [...],
  "annotations": [...],
  "categories": [...]
}

## Imports

In [48]:
import sys
from pathlib import Path
import json
import shutil

In [49]:
# Set execution root in the project root
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

In [50]:
from src.utils.config.config import Config
from src.detector.yolo_detector import YOLODetector

## Config

In [51]:
config_loader = Config()
cfg = config_loader.load_config()

yolo_model_type = cfg.paths.models / config_loader.get("model", "yolo.pretrained")
yolo_imgsz = config_loader.get("model", "yolo.imgsz")

raw_dataset = cfg.project_root / config_loader.get("paths", "dataset.raw") / "CafeV1"

output_dataset = raw_dataset / "yolo"

In [52]:
ALLOWED_CLASSES = [
    0,   # person
    63,  # laptop
    67,  # cell phone
    73   # book
]
YOLO_TO_DATASET_MAP = {
    0 : 1, # person
    63 : 2, # laptop
    67 : 3, # cell phone
    73 : 4 # book
}

## Init YOLO

In [53]:
detector = YOLODetector(yolo_model_type, cfg.project_root)

YOLOv8m summary: 169 layers, 25,902,640 parameters, 0 gradients, 79.3 GFLOPs


## Functions

### load_existing_json

In [54]:
def load_existing_json(json_path):
    if json_path.exists():
        with open(json_path, "r") as f:
            return json.load(f)
    # Empty data
    return None

### save_json

In [55]:
def save_json(data, json_path):
    with open(json_path, "w") as f:
        json.dump(data, f, indent=4)

### process_image

In [56]:
def process_image(image_path, image_id, annotation_id):
    # Do inference on the frame
    results = detector.model(
        source=image_path,
        verbose=False,
        imgsz = yolo_imgsz, # 640, 960, 1280,
        classes = ALLOWED_CLASSES,
        conf=0.15
    )[0]

    # YOLO built-in visualizer (used to verify)
    # results.show()

    detections = []
    if results.boxes is None:
        return detections

    boxes = results.boxes.xyxy.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()

    for i in range(len(boxes)):
        bbox = boxes[i]
        cls = int(classes[i])
        x1, y1, x2, y2 = bbox.tolist() 
        w = x2 - x1
        h = y2 - y1
        
        dataset_cls = YOLO_TO_DATASET_MAP[cls]

        # Format into JSON
        detections.append({
            "id": annotation_id,
            "image_id": image_id,
            "category_id": dataset_cls,
            "bbox": [x1, y1, w, h],
            "area": w * h,
            "iscrowd": 0,
            "attributes": {}
        })
        annotation_id+=1
        
    return detections, annotation_id

### process_clip

In [57]:
def process_clip(clip_path, viewpoint_no):
    clip_name = clip_path.name # 0
    images_path = clip_path / "images" # path to frames dir
    
    output_path = output_dataset / "clips" / viewpoint_no / clip_name
    output_images_path = output_path / "images"  
    output_json_path = output_path / "anns.json"
    
    if not images_path.exists():
        print(f"Couldn't process clip {clip_name}")
        return

    # Get and sort frames
    image_files = (p for p in images_path.iterdir()) # list of frames
    image_files = sorted(image_files, key=lambda x: int(x.stem.split("_")[1]))

    # Ensure path exists
    output_images_path.mkdir(parents=True, exist_ok=True)

    json_data = load_existing_json(output_json_path)
    
    # Create default JSON if not found
    if json_data == None:
        json_data = {
            "images": [],
            "annotations": [],
            "categories": [
                {"id": YOLO_TO_DATASET_MAP[0], "name": "person"},
                {"id": YOLO_TO_DATASET_MAP[63], "name": "laptop"},
                {"id": YOLO_TO_DATASET_MAP[67], "name": "cell phone"},
                {"id": YOLO_TO_DATASET_MAP[73], "name": "book"}
            ]
        }
        
    annotation_id = 0
    
    # frames that have already been processed
    processed_frames = {
        f["id"] 
        for f in (json_data.get("images") or [])
    }
    
    for image_id, image_file in enumerate(image_files):
        # Skip already processed frames
        if image_id in processed_frames:
            continue 
        
        image_path = images_path / image_file
        
        # Add image entry
        json_data["images"].append({
            "id": image_id,
            "file_name": image_file.name,
            "width": 1920,
            "height": 1080
        })

        detections, latest_annotation_id = process_image(image_path, image_id, annotation_id)
        annotation_id = latest_annotation_id

        # Add detections data for current frame
        json_data["annotations"].extend(detections)
        
        # Save frame image
        shutil.copy2(image_path, output_images_path)
        # break # temp - used to validate one frame

    save_json(json_data, output_json_path)
    print(f"Saved anns for clip {clip_name} at {output_json_path}")

## Run

In [58]:
def main():
    clips_path = raw_dataset / "Clips"

    viewpoint_dirs = sorted(
        [p for p in clips_path.iterdir() if p.is_dir()],
        key=lambda p: int(p.name)
    )

    for viewpoint_path in viewpoint_dirs:
        print(f"Accessing viewpoint: {viewpoint_path.name}")

        clip_dirs = sorted(
            [p for p in viewpoint_path.iterdir() if p.is_dir()],
            key=lambda p: int(p.name)
        )
        
        for clip_path in clip_dirs:
            process_clip(clip_path, viewpoint_path.name)
            # break # temp - used to validate one clip
    
        # break # temp - used to validate one viewpoint
    print("All images annotated")
main()

Accessing viewpoint: 1
Saved anns for clip 0 at C:\Research\Dataset\data\raw\CafeV1\yolo\clips\1\0\anns.json
Saved anns for clip 1 at C:\Research\Dataset\data\raw\CafeV1\yolo\clips\1\1\anns.json
Saved anns for clip 2 at C:\Research\Dataset\data\raw\CafeV1\yolo\clips\1\2\anns.json
Saved anns for clip 19 at C:\Research\Dataset\data\raw\CafeV1\yolo\clips\1\19\anns.json
Saved anns for clip 20 at C:\Research\Dataset\data\raw\CafeV1\yolo\clips\1\20\anns.json
Saved anns for clip 21 at C:\Research\Dataset\data\raw\CafeV1\yolo\clips\1\21\anns.json
Saved anns for clip 22 at C:\Research\Dataset\data\raw\CafeV1\yolo\clips\1\22\anns.json
Saved anns for clip 450 at C:\Research\Dataset\data\raw\CafeV1\yolo\clips\1\450\anns.json
Saved anns for clip 451 at C:\Research\Dataset\data\raw\CafeV1\yolo\clips\1\451\anns.json
Accessing viewpoint: 2
Saved anns for clip 0 at C:\Research\Dataset\data\raw\CafeV1\yolo\clips\2\0\anns.json
Saved anns for clip 1 at C:\Research\Dataset\data\raw\CafeV1\yolo\clips\2\1\an